<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex01-python-fundamentals/Ex01_02_numpy_and_broadcasting_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_1 · Notebook 02 — NumPy, and the shape of everything

Arrays against lists, dtype and shape, indexing and slicing, **broadcasting**,
and vectorisation instead of loops with the timing to prove it.

## Why this notebook is longer than the others

Because broadcasting is the single highest-value topic in Part 1, and it is not
close.

From Ex_7 onwards you will be writing physics residuals — expressions that
combine a network's output, its derivatives, and a set of coordinates, and that
must come out with exactly the right shape or the loss you minimise is not the
loss you meant. Essentially every shape bug you will hit in those exercises is
a broadcasting misunderstanding wearing a disguise, and the reason they cost so
much is that broadcasting mostly does **not** raise an error. It quietly
produces an array of a different shape from the one you intended, and the code
carries on.

The specific failure, which you will meet twice in this notebook and probably
twenty times this semester, is the difference between an array of shape `(N,)`
and one of shape `(N, 1)`. Subtract one from the other and NumPy hands you an
`(N, N)` matrix without complaint. Take the mean of that matrix and you get a
number — a perfectly reasonable-looking number, of the wrong quantity, with no
warning anywhere.

Twenty minutes here saves an exercise session in week seven. That is not a
figure of speech; it is what happened to the previous cohort.

## The habit this notebook is trying to install

**Print the shape.** Not sometimes — every time an array is built, reshaped,
indexed or combined, until doing so is automatic. `print(a.shape)` costs
nothing and it is the only way to see a bug that does not raise.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_1_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex01-python-fundamentals/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup -------------------------------------------------------------
# Needs Ex_1_core.py alongside this notebook.
import os
for f in ("Ex_1_core.py",):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from Ex_1_core import *                             # noqa: F401,F403
import numpy as np

set_seed(88)
np.set_printoptions(precision=4, suppress=True)
print("setup complete")

## 1 · An array is not a list

A Python list is a sequence of pointers to arbitrary objects. It may hold an
integer, a string and another list at once, each stored separately, each with
its own type information. That flexibility is exactly what you want for
bookkeeping and exactly what you do not want for numerics.

A NumPy array is a **single contiguous block of memory** holding elements that
all have the same type, together with a shape that says how to interpret that
block as a grid. Three consequences follow, and they are the whole reason NumPy
exists:

1. **Arithmetic is elementwise and it is fast.** `a * 2` doubles every element,
   in a loop written in C, over memory that is already laid out in order.
2. **The type is fixed at creation.** You cannot put a string in a float array.
3. **The shape is an attribute you can and must inspect.**

Watch what `*` does to each of the two. On a list it *repeats* — `[1, 2] * 2`
is `[1, 2, 1, 2]` — and on an array it multiplies. Neither is wrong; they are
different objects with different meanings for the same symbol, and confusing
them produces a list of the wrong length rather than an error.

In [ ]:
py_list = [1.0, 2.0, 3.0, 4.0]
np_array = np.array([1.0, 2.0, 3.0, 4.0])

print("list  * 2  ->", py_list * 2,  "   repeated")
print("array * 2  ->", np_array * 2, "   multiplied")
print()
print("list  + list  ->", py_list + [9.0], "   concatenated")
print("array + 9     ->", np_array + 9.0,  "   added to every element")
print()
describe(np_array, "np_array")

print()
print("a list of 1,000,000 floats costs about", 1_000_000 * 8 + 1_000_000 * 8, "bytes of pointers and objects,")
print("an array of the same numbers costs exactly", np.zeros(1_000_000).nbytes, "bytes")

**What you should see.** The list repeated to eight entries, the array doubled;
the list concatenated, the array shifted; then the description of the array,
with `shape (4,)`, `dtype float64`, 32 bytes. The last two lines are a rough
comparison of memory, and the array wins by a factor that grows with the size
of the problem.

## 2 · dtype and shape

Two attributes carry almost all the information about an array.

**`dtype`** is the element type — `float64` by default for anything built from
floats, `int64` for anything built from Python integers. It is fixed when the
array is created. Assigning a float into an integer array does not promote the
array; it truncates the value, silently. This is the one place where NumPy's
behaviour is genuinely dangerous, and the defence is to build float arrays on
purpose: `np.zeros(n)` is already float, but `np.array([1, 2, 3])` is not.

Deep learning adds a second reason to care. PyTorch defaults to `float32`,
NumPy to `float64`, and mixing them raises in some places and silently
downcasts in others. From Ex_2 you will see `dtype=torch.float32` written
explicitly all over the place, and this is why.

**`shape`** is a tuple giving the length of each axis. Its length is `ndim`.
Read the shape from the left as *outermost first*: an array of shape
`(6, 240)` is six rows of 240 numbers, and the last axis is the one that is
contiguous in memory.

The creation functions you will use every week:

| | |
|---|---|
| `np.array(x)` | from a list or nested list |
| `np.zeros(shape)`, `np.ones(shape)`, `np.full(shape, v)` | filled, float by default |
| `np.arange(a, b, step)` | like `range`, excludes the stop, avoid with floats |
| `np.linspace(a, b, n)` | `n` points **including both ends** — use this for coordinates |
| `np.random.normal(size=shape)` | Gaussian noise |

`arange` with a float step is a trap: floating-point accumulation decides how
many points you get, so `np.arange(0, 1, 0.1)` may give ten or eleven. Use
`linspace` whenever the endpoints matter, which for a spatial grid is always.

In [ ]:
ints = np.array([1, 2, 3])
floats = np.array([1.0, 2.0, 3.0])

print("ints   dtype", ints.dtype, "  floats dtype", floats.dtype)

ints[0] = 2.7                       # no error, no warning
print("after ints[0] = 2.7 :", ints, "  <- truncated to 2, silently")

print()
grid = np.linspace(0.0, 2.0, 5)
print("linspace(0, 2, 5) :", grid, " both ends included, 5 points")
print("arange(0, 2, 0.5) :", np.arange(0.0, 2.0, 0.5), " stop excluded")

print()
for a, name in [(np.zeros((2, 3)), "np.zeros((2, 3))"),
                (np.ones(4), "np.ones(4)"),
                (np.full((2, 2), 9.5), "np.full((2, 2), 9.5)"),
                (np.eye(3), "np.eye(3)")]:
    print(f"  {name:<24} shape {str(a.shape):<10} dtype {a.dtype}")

**What you should see.** `int64` and `float64`; the assignment of 2.7 leaving a
2 behind; `linspace` giving `[0. 0.5 1. 1.5 2. ]` and `arange` stopping at 1.5;
then the four shapes.

The truncation line is the one to remember. Nothing warned you.

### Your turn: build four arrays

Build these four, with exactly these names, shapes and contents:

| name | requirement |
|---|---|
| `x_grid` | 41 points from 0.0 to 2.0 **inclusive**, shape `(41,)` |
| `zeros_2x5` | a 2 by 5 array of zeros, `float64` |
| `counts` | the integers 0 to 5 inclusive, dtype `int64`, shape `(6,)` |
| `noise` | Gaussian noise, mean 0, standard deviation 1, shape `(3, 4)` |

Two of them have a decision in them. For `x_grid`, "inclusive" rules out
`arange` and points you at `linspace` — and note that 41 points across an
interval of 2.0 gives a spacing of exactly 0.05, because `linspace` divides by
`n - 1` and not by `n`. That off-by-one is the reason a grid of 41 points is a
more natural request than one of 40.

For `noise`, use `np.random.normal(size=...)`. The cell already calls
`set_seed(88)` above the TODO, so the numbers you draw are the same on every
run and the same as everyone else's — which is what makes the printed mean and
standard deviation worth comparing with a neighbour.

Four lines.

In [ ]:
set_seed(88)          # so that `noise` is reproducible for the check below

# TODO 1 --- four arrays --------------------------------------------------------
# Four `...` to replace, one per line:
#   x_grid     ->  np.linspace(0.0, 2.0, 41)
#   zeros_2x5  ->  np.zeros((2, 5))
#   counts     ->  np.arange(6)
#   noise      ->  np.random.normal(size=(3, 4))
x_grid    = ...                                   # <- np.linspace(0.0, 2.0, 41)
zeros_2x5 = ...                                   # <- np.zeros((2, 5))
counts    = ...                                   # <- np.arange(6)
noise     = ...                                   # <- np.random.normal(size=(3, 4))
assert not any(v is ... for v in (x_grid, zeros_2x5, counts, noise)), "TODO 1: replace the four ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("x_grid", x_grid, (41,))
check("x_grid endpoints", [x_grid[0], x_grid[-1]], [0.0, 2.0])
check("x_grid spacing", np.diff(x_grid), np.full(40, 0.05), tol=1e-12)

check_shape("zeros_2x5", zeros_2x5, (2, 5))
print("  zeros_2x5 dtype:", zeros_2x5.dtype, "(expected float64)")

check_shape("counts", counts, (6,))
check("counts values", counts, [0, 1, 2, 3, 4, 5])
print("  counts dtype:", counts.dtype, "(expected int64)")

check_shape("noise", noise, (3, 4))
print(f"  noise mean {noise.mean():+.3f}, std {noise.std():.3f} "
      f"(a 12-sample estimate, so do not expect 0.000 and 1.000)")

**What you should see.** `PASS` on every shape and value check. The noise mean
and standard deviation will be somewhere near 0 and 1 but not equal to them —
twelve samples is not many, and being unsurprised by that is part of the point.

## 3 · Indexing, slicing, and the fact that a slice is a view

Indexing a 2-D array takes one index per axis, inside a single pair of
brackets: `a[1, 2]`, not `a[1][2]`. The second form works but goes through an
intermediate array and does not generalise to slices.

A slice `start:stop:step` follows the same rule as a list slice — the stop is
excluded — and a bare `:` means the whole axis. So `a[:, 0]` is the first
column and `a[0, :]` is the first row, and both come back as 1-D arrays of
shape `(n,)`. **Indexing with an integer removes an axis.** Slicing with a
colon keeps it. `a[0:1, :]` and `a[0, :]` contain the same numbers with shapes
`(1, m)` and `(m,)`, and that difference is the seed of the broadcasting bug in
§4.

Now the part that surprises people: **a basic slice is a view, not a copy.** It
shares memory with the parent array. Write into the slice and the parent
changes. This is what makes NumPy fast — no data is moved — and it is a real
source of bugs when you assume you have an independent working copy. If you
want one, say `.copy()`.

**Boolean masks** and **fancy indexing** are the two forms that *do* copy.
`a[a > 3]` selects the elements where the condition holds, flattening the
result. `a[[0, 2]]` selects rows 0 and 2 in the order given. Boolean masking is
how you should express "the points on the boundary" or "the samples above the
threshold" — a loop with an `if` inside does the same job an order of magnitude
slower and reads worse.

In [ ]:
a = np.arange(12.0).reshape(3, 4)
print("a =")
print(a)

print()
print("a[1, 2]     ", a[1, 2],      "   one element, a scalar")
print("a[0, :]     ", a[0, :],      "shape", a[0, :].shape,   " row, axis removed")
print("a[0:1, :]   ", a[0:1, :],    "shape", a[0:1, :].shape, " row, axis kept")
print("a[:, 1]     ", a[:, 1],      "shape", a[:, 1].shape)
print("a[-1, -1]   ", a[-1, -1],    "   bottom right")

print()
row = a[0, :]              # a VIEW
row[0] = -999.0
print("after writing into the view, a[0, 0] =", a[0, 0], " <- the parent changed")

a = np.arange(12.0).reshape(3, 4)
row_copy = a[0, :].copy()
row_copy[0] = -999.0
print("after writing into the copy, a[0, 0] =", a[0, 0], "  <- unchanged")

print()
mask = a > 6.0
print("mask =")
print(mask)
print("a[mask]      ", a[mask], "  flattened, and it is a copy")
print("count above 6:", mask.sum(), " because True counts as 1")
print("np.where row 1", np.where(a > 6.0, 1.0, 0.0)[1], " element-wise choice")

**What you should see.** The 3 by 4 grid; `a[0, :]` with shape `(4,)` against
`a[0:1, :]` with shape `(1, 4)`; then the view writing through to the parent
and the copy not doing so; then the mask, the five elements above 6, and the
count.

The mask counting trick — `mask.sum()` — works because `True` is 1. You will
use it constantly to answer "how many points violated this?".

### Your turn: index a sensor array

`sensor_matrix()` returns a `(6, 240)` array of strain-gauge readings in
microstrain: six gauges, 240 samples each, one gauge per row. It also returns
the true per-gauge offsets and gains, which you will need in §4.

Produce four things:

| name | requirement |
|---|---|
| `gauge_3` | the whole of row 3, as a 1-D array of shape `(240,)` |
| `first_10` | the first ten samples of every gauge, shape `(6, 10)` |
| `last_sample` | the final sample from every gauge, shape `(6,)` |
| `big_readings` | every reading anywhere in the array whose magnitude exceeds 150 microstrain, as a 1-D array |

For `big_readings`, "magnitude" means absolute value, so the condition involves
`np.abs`. Build the mask and index with it in one expression; the result is 1-D
however the mask was shaped, which is the normal and useful behaviour.

Row 3 means index 3, which is the fourth gauge. Counting from zero is not
optional and this is a good place to feel it.

In [ ]:
readings, true_offsets, true_gains = sensor_matrix()
describe(readings, "readings")

# TODO 2 --- index the sensor array ----------------------------------------------
# Four `...` to replace, one per line:
#   gauge_3       ->  readings[3]                      row 3, shape (240,)
#   first_10      ->  readings[:, :10]                 all rows, first 10 columns
#   last_sample   ->  readings[:, -1]                  all rows, last column
#   big_readings  ->  readings[np.abs(readings) > 150.0]   boolean mask, 1-D result
gauge_3      = ...                                # <- readings[3]
first_10     = ...                                # <- readings[:, :10]
last_sample  = ...                                # <- readings[:, -1]
big_readings = ...                                # <- readings[np.abs(readings) > 150.0]
assert not any(v is ... for v in (gauge_3, first_10, last_sample, big_readings)), "TODO 2: replace the four ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("gauge_3", gauge_3, (240,))
check("gauge_3 is row 3", gauge_3, readings[3])
check_shape("first_10", first_10, (6, 10))
check_shape("last_sample", last_sample, (6,))
check("last_sample values", last_sample, readings[:, -1])
print(f"  big_readings: {big_readings.size} readings exceed 150 microstrain "
      f"out of {readings.size}")
check("big_readings", np.sort(big_readings),
      np.sort(readings[np.abs(readings) > 150.0]))

**What you should see.** Every check `PASS`, and 63 large readings out of
1,440. If `gauge_3` has shape `(1, 240)` you sliced where
you should have indexed — `readings[3:4]` keeps the axis, `readings[3]` removes
it.

## 4 · Broadcasting

Here it is. Read this section slowly; the rest of the course rests on it.

### The problem it solves

You have temperatures at 100 positions, shape `(100,)`, and you want to
subtract a single reference temperature. You have six gauges by 240 samples,
shape `(6, 240)`, and you want to subtract a different offset from each gauge —
six numbers. Neither of these should require a loop, and neither should require
you to build a full-size array of repeated values.

Broadcasting is the rule that lets an array of one shape take part in an
operation with an array of another, by treating a length-1 axis as if it were
repeated. Nothing is actually copied; NumPy walks the memory with a stride of
zero along the stretched axis. That is why it is fast as well as convenient.

### The rule, stated once

Two shapes are compatible if, **aligning them from the right**, every pair of
lengths is either equal or one of them is 1. Missing axes on the left of the
shorter shape are treated as 1.

That is the whole rule. There is no second clause. Everything else is
consequence:

- `(6, 240)` with `(240,)`: align right. 240 with 240, same. Then 6 with
  nothing, treated as 1, stretched. Result `(6, 240)`. The one-dimensional
  array is applied to every row.
- `(6, 240)` with `(6, 1)`: 240 with 1, stretch. 6 with 6, same. Result
  `(6, 240)`. One number per row, which is what you wanted.
- `(6, 240)` with `(6,)`: align right. 240 with 6. Not equal, neither is 1.
  **ValueError.** This is the error people find infuriating, and it is the rule
  protecting you: NumPy has no way to know you meant one value per row rather
  than per sample. Say so, by reshaping to `(6, 1)`.
- `(100,)` with `(100, 1)`: align right. 100 with 1, stretch. Then nothing with
  100, stretch. Result `(100, 100)`. **No error.** This is the dangerous one.

`broadcast_table` in the core module prints exactly this alignment. Use it
whenever a shape surprises you.

In [ ]:
banner("(6, 240) and (240,)  -  one value per sample")
broadcast_table((6, 240), (240,))

print()
banner("(6, 240) and (6, 1)  -  one value per gauge")
broadcast_table((6, 240), (6, 1))

print()
banner("(6, 240) and (6,)  -  what people write when they mean the line above")
broadcast_table((6, 240), (6,))

print()
banner("(100,) and (100, 1)  -  succeeds, and is almost never what you meant")
broadcast_table((100,), (100, 1))

**What you should see.** Four alignment tables. The third ends in `CLASH` and a
`ValueError`; the fourth succeeds and produces `(100, 100)`.

Sit with the contrast between the last two for a moment. The one that raises is
the *safe* case: you get an error message, at the line where you made the
mistake, and you fix it in thirty seconds. The one that succeeds is the case
that costs an afternoon.

### How to add an axis

Three notations, all common, all meaning the same thing:

```python
v[:, None]           # None inserts a new axis of length 1
v[:, np.newaxis]     # np.newaxis is literally None, spelled out
v.reshape(-1, 1)     # -1 means "work it out from the total size"
```

Use whichever you find most readable — this course uses `[:, None]` in prose
and `reshape(-1, 1)` when the intent is "make this a column". `v[None, :]`
makes a row of shape `(1, n)`.

And in the other direction, reductions take an `axis` argument and a `keepdims`
flag:

```python
a.mean(axis=1)                  # shape (6,)     - the axis is removed
a.mean(axis=1, keepdims=True)   # shape (6, 1)   - the axis is kept, as length 1
```

`keepdims=True` exists precisely so that the result broadcasts back against the
array you reduced. `a - a.mean(axis=1, keepdims=True)` centres each row and has
the same shape as `a`. Without `keepdims` that expression raises, which is the
good outcome, or silently transposes your meaning if the array happens to be
square, which is not.

**Remember which axis goes.** `axis=0` collapses axis 0, so summing a
`(6, 240)` array along `axis=0` leaves `(240,)` — one number per sample, added
across the gauges. It is the axis you *name* that disappears.

In [ ]:
v = np.array([1.0, 2.0, 3.0])
print("v            shape", v.shape)
print("v[:, None]   shape", v[:, None].shape,  " a column")
print("v[None, :]   shape", v[None, :].shape,  " a row")
print("v.reshape(-1, 1) shape", v.reshape(-1, 1).shape)

print()
a = np.arange(12.0).reshape(3, 4)
print("a               shape", a.shape)
print("a.sum(axis=0)   shape", a.sum(axis=0).shape, a.sum(axis=0))
print("a.sum(axis=1)   shape", a.sum(axis=1).shape, a.sum(axis=1))
print("a.sum(axis=1, keepdims=True) shape", a.sum(axis=1, keepdims=True).shape)

print()
print("row-centred, with keepdims:")
print(a - a.mean(axis=1, keepdims=True))

**What you should see.** The three ways of making a column or a row; then
`axis=0` giving four numbers and `axis=1` giving three, confirming that the
named axis is the one removed; then a row-centred array whose every row sums to
zero.

### Your turn 1: predict the shapes

Before writing any more code, predict. Fill in the dictionary `predictions`
below: for each pair of shapes, give the shape of the result as a **tuple**, or
the string `"error"` if the operation raises.

Do it by applying the rule in your head — align from the right, pad the shorter
with 1s, every pair must be equal or contain a 1 — and only then run the check.
Guessing and correcting teaches you very little; predicting and being wrong
teaches you a great deal, because you find out exactly which clause of the rule
you had wrong.

The pairs, as keys of the dictionary:

```
("a", (3, 4),    (4,))
("b", (3, 4),    (3,))
("c", (3, 4),    (3, 1))
("d", (5,),      (5, 1))
("e", (2, 1, 4), (3, 4))
("f", (6, 240),  (1, 240))
("g", (2, 3),    (2, 3))
("h", (10, 1),   (1, 7))
```

Case (e) is the only one that needs real care, and it is worth doing on paper:
align `(2, 1, 4)` and `(3, 4)` from the right, pad the second to `(1, 3, 4)`,
and then compare axis by axis.

Answer with a dictionary literal:

```python
predictions = {"a": (3, 4), "b": "error", ...}
```

In [ ]:
# TODO 3 --- predict the broadcast shapes -----------------------------------------
# Eight `...` to replace. Each is either a shape tuple such as (3, 4)
# or the string "error". Work them out with the rule above; the check tells
# you which ones you got wrong.
predictions = {
    "a": ...,        # (3, 4)    with (4,)
    "b": ...,        # (3, 4)    with (3,)
    "c": ...,        # (3, 4)    with (3, 1)
    "d": ...,        # (5,)      with (5, 1)
    "e": ...,        # (2, 1, 4) with (3, 4)
    "f": ...,        # (6, 240)  with (1, 240)
    "g": ...,        # (2, 3)    with (2, 3)
    "h": ...,        # (10, 1)   with (1, 7)
}
assert not any(v is ... for v in predictions.values()), "TODO 3: replace the eight ..."
# ------------------------------------------------------------------------------

In [ ]:
cases = {
    "a": ((3, 4),    (4,)),
    "b": ((3, 4),    (3,)),
    "c": ((3, 4),    (3, 1)),
    "d": ((5,),      (5, 1)),
    "e": ((2, 1, 4), (3, 4)),
    "f": ((6, 240),  (1, 240)),
    "g": ((2, 3),    (2, 3)),
    "h": ((10, 1),   (1, 7)),
}

score = 0
for key, (sa, sb) in cases.items():
    try:
        truth = (np.zeros(sa) + np.zeros(sb)).shape
    except ValueError:
        truth = "error"
    guess = predictions.get(key, "MISSING")
    ok = (guess == truth)
    score += ok
    print(f"  {'PASS' if ok else 'FAIL'}  {key}: {sa} with {sb}"
          f"   you said {guess},  NumPy says {truth}")

print(f"\n  {score} of {len(cases)} correct")
print("\n  the alignment for any you got wrong:")
for key, (sa, sb) in cases.items():
    if predictions.get(key) != ((np.zeros(sa) + np.zeros(sb)).shape
                                if can_broadcast(sa, sb) else "error"):
        print(f"\n  case {key}")
        broadcast_table(sa, sb)

**What you should see.** Eight verdicts and a score, then the alignment table
for every case you got wrong. If you scored 8 the last section prints nothing.

The two most often missed are (b), which raises, and (d), which does not. They
are the same mistake seen from two sides: NumPy will not guess that a length-3
vector was meant to apply per row, and it will happily turn a length-5 vector
and a 5 by 1 column into a 5 by 5 matrix.

### Your turn 2: calibrate the gauges

Now use it for something real.

`readings` is `(6, 240)`. Each gauge has an unknown zero offset and an unknown
gain error, so what you have is roughly

    reading = gain × true signal + offset

You do not know the gains, but you can remove the offsets from the data alone,
because the true signal has zero mean over the record while the offset does
not. So:

1. Compute `row_means`, the mean of each gauge over its 240 samples, with shape
   **`(6, 1)`**. Use `axis=` and `keepdims=True`.
2. Compute `centred = readings - row_means`. The subtraction is the
   broadcasting: `(6, 240)` against `(6, 1)`, stretching the 1 across all 240
   samples.
3. Compute `normalised`, in which each gauge is additionally divided by its own
   standard deviation, so that every row has zero mean and unit standard
   deviation. Use `keepdims=True` again.

Three lines. No loop over gauges — if you write `for i in range(6)` here you
have missed the point of the section, and more importantly you have written
something that will be a hundred times slower when the six gauges become six
thousand nodes of a finite-element mesh.

The check verifies that the recovered offsets match the true ones the generator
used, to within the noise, and that each row of `normalised` really does have
unit standard deviation.

In [ ]:
# `readings`, `true_offsets`, `true_gains` come from the cell in section 3

# TODO 4 --- calibrate the gauges ------------------------------------------------
# Three `...` to replace, one per line:
#   row_means   ->  readings.mean(axis=1, keepdims=True)         shape (6, 1)
#   centred     ->  readings - row_means                          shape (6, 240)
#   normalised  ->  centred / centred.std(axis=1, keepdims=True)  shape (6, 240)
row_means  = ...                                  # <- readings.mean(axis=1, keepdims=True)
centred    = ...                                  # <- readings - row_means
normalised = ...                                  # <- centred / centred.std(axis=1, keepdims=True)
assert not any(v is ... for v in (row_means, centred, normalised)), "TODO 4: replace the three ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("row_means", row_means, (6, 1))
check_shape("centred", centred, (6, 240))
check_shape("normalised", normalised, (6, 240))

check("centred rows have zero mean", centred.mean(axis=1), np.zeros(6), tol=1e-10)
check("normalised rows have unit std", normalised.std(axis=1), np.ones(6), tol=1e-10)

print()
print("  recovered offset vs the true one used by the generator, in microstrain:")
for i in range(6):
    print(f"    gauge {i}: recovered {row_means[i, 0]:+8.2f}   true {true_offsets[i, 0]:+8.2f}")
check("offsets recovered to within the noise", row_means, true_offsets, tol=2.0)

**What you should see.** Every shape and statistic `PASS`, and six pairs of
numbers agreeing to within a microstrain or two. The recovery is not exact
because the mean of a finite noisy record is not exactly the offset, which is
why the tolerance on the last check is 2.0 rather than 1e-10. Tolerances that
match the physics rather than the arithmetic are a recurring theme from Ex_7
onwards.

If `centred` came out with shape `(6, 6)` or the subtraction raised, you left
out `keepdims=True`.

### Your turn 3: the `(N,)` versus `(N, 1)` trap, deliberately

This is the exercise the whole notebook exists for.

You have a set of measured deflections at 25 positions along the beam, and the
model's prediction at the same positions. You want the **residual** — measured
minus predicted, one number per position, shape `(25,)` — and then the root
mean square error, one number.

The trap is that in real code these two arrays often arrive with different
shapes. A prediction from a neural network comes out as `(25, 1)`, because a
network with one output produces a column. A measurement loaded from a file is
`(25,)`. Subtract them directly and you get a `(25, 25)` matrix of every
measurement minus every prediction, whose mean square root is a number that
looks entirely plausible and means nothing.

The cell below sets up exactly that situation and shows you the wrong answer.
Then your task:

1. Compute `residual_wrong = measured - predicted_col` and record its shape and
   RMS, just to see the size of the lie.
2. Compute `residual_right`, of shape `(25,)`, by making the two arrays agree
   in shape first. Either flatten the column with `.ravel()` or promote the
   measurement with `[:, None]` and then flatten the result — but be explicit
   about which you did.
3. Compute `rms_right`, the root mean square of `residual_right`, as a float.

The interesting part is not the code, which is two lines. It is the comparison
between the two RMS values, and the fact that nothing in Python told you which
one to trust.

In [ ]:
x_meas = np.linspace(0.0, 2.0, 25)
measured = measured_deflection(x_meas, noise_mm=0.04, seed=88)   # shape (25,)
predicted_col = beam_deflection(x_meas).reshape(-1, 1)           # shape (25, 1)

describe(measured, "measured")
describe(predicted_col, "predicted_col")

wrong = measured - predicted_col
print()
print("  measured - predicted_col has shape", wrong.shape,
      "and its RMS is", f"{np.sqrt((wrong ** 2).mean()):.4f} mm")
print("  ... which is not an error, and is not the answer to any question you asked")

In [ ]:
# TODO 5 --- the (N,) versus (N, 1) trap ------------------------------------------
# Three `...` to replace, one per line:
#   residual_wrong  ->  measured - predicted_col            shape (25, 25), the lie
#   residual_right  ->  measured - predicted_col.ravel()    shape (25,)
#   rms_right       ->  float(np.sqrt((residual_right ** 2).mean()))
residual_wrong = ...                              # <- measured - predicted_col
residual_right = ...                              # <- measured - predicted_col.ravel()
rms_right      = ...                              # <- float(np.sqrt((residual_right ** 2).mean()))
assert not any(v is ... for v in (residual_wrong, residual_right, rms_right)), "TODO 5: replace the three ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("residual_right", residual_right, (25,))
truth = measured - beam_deflection(x_meas)
check("residual_right", residual_right, truth, tol=1e-12)
check("rms_right", rms_right, float(np.sqrt((truth ** 2).mean())), tol=1e-12)

print()
print(f"  RMS from the (25, 25) matrix : {np.sqrt((residual_wrong ** 2).mean()):.4f} mm")
print(f"  RMS from the (25,) residual  : {rms_right:.4f} mm")
print(f"  the noise actually added was : 0.0400 mm")
print()
print("  the second one recovers the instrument noise. the first does not, and")
print("  nothing about it looked wrong.")

**What you should see.** Three `PASS` lines, then the two RMS values. The
correct one lands near 0.04 mm, recovering the noise that was added. The
incorrect one is roughly an order of magnitude larger, because the matrix is
dominated by comparing a measurement at one end of the beam with a prediction
at the other.

Take the lesson in its general form: **an operation that succeeds is not an
operation that is right.** The only defence is to know the shape you expect
before you write the line, and to check it afterwards.

### Your turn 4: a distance matrix, where the (N, N) is the point

To be fair to broadcasting, the behaviour that traps you in §3 is the same
behaviour that makes a whole class of computation trivial. When you *do* want
every pair, broadcasting gives it to you in one line.

Given positions `p` of shape `(N,)`, build `dist`, the `(N, N)` matrix whose
entry `(i, j)` is `|p[i] - p[j]|`.

The construction is exactly the one you were warned about, used on purpose:
subtract a column from a row. `p[:, None]` has shape `(N, 1)`, `p[None, :]` has
shape `(1, N)`, and their difference has shape `(N, N)` with entry `(i, j)`
equal to `p[i] - p[j]`. Take the absolute value.

One line. Then convince yourself the result is symmetric with a zero diagonal,
which the check does for you.

This is the same construction that appears in Ex_5 as the adjacency of a graph
and in Ex_9 as a kernel between collocation points. It is worth doing once
here, deliberately, so that the `(N, N)` never again appears by accident.

In [ ]:
p = np.array([0.0, 0.5, 1.25, 2.0])

# TODO 6 --- a distance matrix, on purpose -----------------------------------------
# One `...` to replace:  np.abs(p[:, None] - p[None, :])
#   p[:, None] is a (4, 1) column, p[None, :] a (1, 4) row, the difference (4, 4)
dist = ...                                        # <- np.abs(p[:, None] - p[None, :])
assert dist is not ..., "TODO 6: replace the ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("dist", dist, (4, 4))
check("zero diagonal", np.diag(dist), np.zeros(4))
check("symmetric", dist, dist.T)
check("dist[0, 3]", dist[0, 3], 2.0)
check("dist[1, 2]", dist[1, 2], 0.75)
print()
print(dist)

**What you should see.** Five `PASS` lines and a symmetric 4 by 4 matrix with
zeros on the diagonal.

## 5 · Vectorisation, and what it is worth

Every operation so far has been *vectorised*: written as a whole-array
expression rather than as a loop over elements. The reason is not elegance. It
is that the loop happens in compiled C over contiguous memory instead of in the
Python interpreter, and the interpreter's overhead per iteration — unboxing an
object, looking up a method, allocating a result — dwarfs the arithmetic.

The factor is usually between 20 and 200. It is worth measuring rather than
believing, so measure it.

The task below computes, for 200,000 positions, the beam deflection formula

    y(x) = w x² (6L² − 4Lx + x²) / (24 E I)

three ways: a Python `for` loop with `append`, a list comprehension, and a
NumPy expression. `time_call` runs each one five times and keeps the fastest,
which is the fairest single number available on a shared machine.

Write `deflect_numpy(x)`, taking an array and returning an array, as a single
vectorised expression. It is the same formula with no loop anywhere. Then run
the timing cell.

One warning about interpreting the result: the speed-up you measure depends on
the machine, on whether Colab has given you a busy core, and on the size of the
array. The *magnitude* is the message, not the digits. If you measure a factor
of 40 and the text says 60, nothing is wrong.

In [ ]:
L, W, E, I = 2.0, 500.0, 210e9, 8.0e-6
x_big = np.linspace(0.0, L, 200_000)

def deflect_loop(x):
    out = []
    for xi in x:
        out.append(W * xi ** 2 * (6 * L ** 2 - 4 * L * xi + xi ** 2) / (24 * E * I) * 1000.0)
    return np.array(out)

def deflect_comprehension(x):
    return np.array([W * xi ** 2 * (6 * L ** 2 - 4 * L * xi + xi ** 2) / (24 * E * I) * 1000.0
                     for xi in x])

print("loop and comprehension defined; the NumPy version is yours to write")

In [ ]:
# TODO 7 --- the same formula, vectorised ------------------------------------------
# One `...` to replace: the formula from deflect_loop with `x` in place of `xi`,
# so that it acts on the whole array at once:
#   W * x ** 2 * (6 * L ** 2 - 4 * L * x + x ** 2) / (24 * E * I) * 1000.0
def deflect_numpy(x):
    """Deflection in mm for an array of positions, as one vectorised expression."""
    return ...                                    # <- W * x ** 2 * (6 * L ** 2 - 4 * L * x + x ** 2) / (24 * E * I) * 1000.0
# ------------------------------------------------------------------------------

In [ ]:
check("deflect_numpy agrees with the loop",
      deflect_numpy(x_big[:1000]), deflect_loop(x_big[:1000]), tol=1e-9)
check("deflect_numpy agrees with beam_deflection",
      deflect_numpy(x_big[:1000]), beam_deflection(x_big[:1000]), tol=1e-9)

print()
timings = [
    ("python for loop",   time_call(deflect_loop, x_big, repeat=3)),
    ("list comprehension", time_call(deflect_comprehension, x_big, repeat=3)),
    ("numpy, vectorised",  time_call(deflect_numpy, x_big, repeat=5)),
]
compare_timings(timings)

**What you should see.** Two `PASS` lines, then a table. On a typical Colab CPU
the loop takes something like 0.1 s for 200,000 points, the comprehension a
little less, and the NumPy expression around a millisecond — a speed-up in the
tens to low hundreds.

Two things worth noticing in that table:

- The list comprehension is barely faster than the explicit loop. It is more
  readable, not fundamentally quicker, because the arithmetic still happens one
  Python object at a time. Vectorisation is not about syntax.
- The gap widens with array size, and it widens further when the operation is
  cheap. For expensive per-element work — calling a solver, say — the
  interpreter overhead matters less and a loop may be perfectly reasonable.

And the caveat that keeps this honest: **vectorise for speed only when speed is
the problem.** A clear loop that runs once is better than a clever expression
nobody can read. What you must not do is write the loop when the vectorised
form is *also* the clearer one, which for elementwise arithmetic it almost
always is.

## 6 · The debugging recipe

When an array expression misbehaves — whether it raised or, worse, did not —
work through this in order. It resolves the great majority of cases in under a
minute.

1. **Print every shape on the line.** `print(a.shape, b.shape)` immediately
   above the failure. Do this before forming any theory.
2. **Say out loud what shape you expected** and why. "One residual per
   collocation point, so `(N,)`." If you cannot state it, that is the bug.
3. **Feed the two shapes to `broadcast_table`.** It tells you which axis
   clashed, or which axis was stretched when you did not want it stretched.
4. **Look for a stray axis of length 1.** It came either from a network output,
   from `keepdims=True`, or from slicing with `0:1` instead of `0`. Remove it
   with `.ravel()`, `.squeeze()` or `.reshape(-1)`.
5. **Check the dtype** if the numbers are wrong rather than the shape.
   Integer arrays truncate. `float32` and `float64` mixed together will bite
   you from Ex_2 onwards.
6. **Test on a tiny array** — four elements, not 200,000 — where you can read
   the answer by eye and check it against arithmetic you did in your head.

## What you have done

You can state the broadcasting rule from memory; you can predict the shape of a
combination before running it; you can add and remove axes deliberately with
`None`, `reshape` and `keepdims`; you have seen the `(N,)` against `(N, 1)`
failure produce a plausible and wrong number; you have used the same
construction on purpose to build a distance matrix; and you have measured what
vectorisation is worth rather than taking it on trust.

That is the whole of the shape machinery you need for the rest of the course.
What remains is practice.

## Next

`Ex01_03_plotting.ipynb` — engineering figures, which is where the arrays you
have been building finally have to communicate something to a reader.